# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

## Step 16 / 18b scratch — baseline OCR numbers (pretrained, not fine-tuned)

Full-book run on Kaggle GPU T4×2 (`facebook/nougat-base`, **not** fine-tuned —
fine-tuning is Sprint 4 / Step 28). `KAGGLE/step18b_ocr_repair/kaggle_step18b_ocr_repair.ipynb`
produced `data/ocr/` (the Step 18b
repaired-reader re-run, 2026-08-10); the cells below read it directly, so every number is
reproducible from this repo's own state, not copied by hand.

Step 16's baseline missed most of the book (15.1% word coverage); Step 18b fixed three
inference-path bugs and re-ran the full book. The gate below is plan.md Step 18b's own
pre-committed decision rule (no-output rate ≤25% → keep the train set; >25% → reopen the
annotation budget, evidence-based) — it did **not** pass, so the train set was expanded
105→122 pages, targeted at the chapters the repaired reader still failed hardest on. See
`plan.md` Step 18b for the full evidence and decision record.

This is the **BEFORE** number Step 29 will compare the fine-tuned reader against.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "../src")  # notebook runs from notebooks/; doc_agent lives in ../src
from doc_agent.eval import metrics

OCR_DIR = Path("../data/ocr")
LABELS_PATH = Path("../grading_kit/labels.jsonl")
N_CONTENT_PAGES = 1040  # ingest/loader.py: as_p* pages after dropping blanks/front matter

failures = json.loads((OCR_DIR / "failures.json").read_text(encoding="utf-8"))
mmd_files = sorted(OCR_DIR.glob("*.mmd"))
by_reason: dict[str, int] = {}
for row in failures:
    by_reason[row["reason"]] = by_reason.get(row["reason"], 0) + 1

print(f"pages attempted      : {N_CONTENT_PAGES}")
print(f"transcripts produced : {len(mmd_files)}")
print(f"failed / degenerate  : {len(failures)}  ({100 * len(failures) / N_CONTENT_PAGES:.1f}%)")
for reason, count in sorted(by_reason.items(), key=lambda kv: -kv[1]):
    print(f"   {reason:<28} {count:>4}  ({100 * count / len(failures):.1f}% of failures)")

words = sum(len(p.read_text(encoding="utf-8").split()) for p in mmd_files)
print(f"\nwords from OUR OCR   : {words}  (task.yaml floor: 60,000 -- {'MET' if words >= 60000 else 'NOT MET'})")

gold: dict[str, str] = {}
for line in LABELS_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line:
        row = json.loads(line)
        gold[row["page_id"]] = row["text"]

print(f"\n{'page':<10} {'char-F1':>9} {'exact-form':>12} {'gold-form':>10} {'pred-form':>10}  status")
print("-" * 72)
failed_reason = {row["page_id"]: row["reason"] for row in failures}
scored = []
for pid in ("as_p0243", "as_p0255", "as_p0360"):
    mmd = OCR_DIR / f"{pid}.mmd"
    if not mmd.exists():
        reason = failed_reason.get(pid, "?")
        print(f"{pid:<10} {'--':>9} {'--':>12} {'--':>10} {'--':>10}  FAILED: {reason}")
        continue
    pred = mmd.read_text(encoding="utf-8")
    g = gold[pid]
    f1 = metrics.ocr_f1(pred, g)
    ex = metrics.exact_formula_match(pred, g)
    ngf = len(metrics.extract_formulas(g))
    npf = len(metrics.extract_formulas(pred))
    print(f"{pid:<10} {f1:>9.4f} {ex:>12.4f} {ngf:>10} {npf:>10}  ok")
    scored.append((pid, f1, ex, ngf))
print("-" * 72)

if scored:
    mean_f1 = sum(s[1] for s in scored) / len(scored)
    total_f = sum(s[3] for s in scored)
    weighted_ex = (
        sum(s[2] * s[3] for s in scored) / total_f if total_f else 0.0
    )
    print(f"mean char-F1 (n={len(scored)} gold page(s) with a transcript): {mean_f1:.4f}")
    print(f"exact-formula-match, weighted by formula count: {weighted_ex:.4f}")

print("\nWe expected  the FINE-TUNED reader to land at char-F1 0.88-0.93,")
print("exact-match 0.55-0.75. This is the pretrained BEFORE number Step 29 compares against.")


pages attempted      : 1040
transcripts produced : 744
failed / degenerate  : 296  (28.5%)
   empty-or-near-empty           175  (59.1% of failures)
   nougat-missing-page-marker     73  (24.7% of failures)
   repetition-degeneration        48  (16.2% of failures)

words from OUR OCR   : 161941  (task.yaml floor: 60,000 -- MET)

page         char-F1   exact-form  gold-form  pred-form  status
------------------------------------------------------------------------


as_p0243      0.3360       0.0000          0          0  ok
as_p0255      0.2446       0.0000         14          3  ok


as_p0360      0.5910       0.0000         13         18  ok
------------------------------------------------------------------------
mean char-F1 (n=3 gold page(s) with a transcript): 0.3905
exact-formula-match, weighted by formula count: 0.0000

We expected  the FINE-TUNED reader to land at char-F1 0.88-0.93,
exact-match 0.55-0.75. This is the pretrained BEFORE number Step 29 compares against.


In [2]:
# --- Step 18b: the BEFORE/AFTER table + the PDF-text-layer coverage metric the cell above
# doesn't compute (that's the diagnostic that actually found this bug -- see plan.md
# Step 18b). BEFORE numbers are Step 16's measured baseline, hardcoded because the pre-repair
# data/ocr/ no longer exists to recompute them from (same pattern as the Kaggle notebook's
# own BEFORE_REPAIR constant). Everything under AFTER is computed live from this repo's
# current data/ocr/, same as the cell above -- reproducible, not copied by hand.
import pymupdf

PDF_PATH = Path("../data/raw/handbookofmathem1964abra.pdf")
FRONT_MATTER_OFFSET = 32  # printed N = PDF N + 32 (scripts/get_data.sh)

BEFORE_REPAIR = {
    "no_output_rate": 0.429,
    "median_page_coverage": 0.28,
    "book_wide_coverage": 0.151,
    "as_p0360_precision": 0.813,
    "as_p0360_recall": 0.280,
}

_pdf_doc = pymupdf.open(PDF_PATH)


def _pdf_word_count(printed_page: int) -> int | None:
    idx = printed_page + FRONT_MATTER_OFFSET - 1
    if idx < 0 or idx >= _pdf_doc.page_count:
        return None
    return len(_pdf_doc.load_page(idx).get_text("text").split())


per_page_coverage = []
for f in mmd_files:
    pid = f.stem
    if not pid.startswith("as_p"):
        continue
    pdf_words = _pdf_word_count(int(pid[4:]))
    if not pdf_words:
        continue
    ocr_words = len(f.read_text(encoding="utf-8").split())
    per_page_coverage.append(ocr_words / pdf_words)
per_page_coverage.sort()
median_coverage = per_page_coverage[len(per_page_coverage) // 2] if per_page_coverage else 0.0

book_wide_pdf_words = sum(
    len(_pdf_doc.load_page(i).get_text("text").split())
    for i in range(FRONT_MATTER_OFFSET, _pdf_doc.page_count)
)
book_wide_coverage = words / book_wide_pdf_words if book_wide_pdf_words else 0.0

# Precision/recall on as_p0360, the flagship diagnostic page (plan.md Step 18b): same LCS
# decomposition ocr_f1 uses internally, exposed because the whole diagnosis rests on this
# split -- 81% precision / 28% recall pre-repair meant "accurate but stops early", not
# "misreads", and F1 alone would hide which of the two actually moved.
_p360_pred = (OCR_DIR / "as_p0360.mmd").read_text(encoding="utf-8")
_p360_gold = gold["as_p0360"]
_p_norm, _g_norm = metrics.normalize_latex(_p360_pred), metrics.normalize_latex(_p360_gold)
_overlap = metrics._lcs_length(_p_norm, _g_norm)
p360_precision = _overlap / len(_p_norm) if _p_norm else 0.0
p360_recall = _overlap / len(_g_norm) if _g_norm else 0.0

no_output_rate = len(failures) / N_CONTENT_PAGES

print("=" * 78)
print("STEP 18B -- THE FOUR BEFORE/AFTER NUMBERS (plan.md's own gate)")
print("=" * 78)
print(f"{'metric':<40}{'BEFORE (Step 16)':>18}{'AFTER (this run)':>18}")
print("-" * 78)
print(
    f"{'no-output rate':<40}"
    f"{100 * BEFORE_REPAIR['no_output_rate']:>17.1f}%{100 * no_output_rate:>17.1f}%"
)
print(
    f"{'median page coverage vs PDF text':<40}"
    f"{100 * BEFORE_REPAIR['median_page_coverage']:>17.1f}%{100 * median_coverage:>17.1f}%"
)
print(
    f"{'book-wide word coverage vs PDF text':<40}"
    f"{100 * BEFORE_REPAIR['book_wide_coverage']:>17.1f}%{100 * book_wide_coverage:>17.1f}%"
)
print(
    f"{'as_p0360 precision':<40}"
    f"{BEFORE_REPAIR['as_p0360_precision']:>18.3f}{p360_precision:>18.3f}"
)
print(
    f"{'as_p0360 recall':<40}"
    f"{BEFORE_REPAIR['as_p0360_recall']:>18.3f}{p360_recall:>18.3f}"
)
print("-" * 78)

gate_passed = no_output_rate <= 0.25
print(f"\nGATE (plan.md Step 18b): no-output rate {'<=' if gate_passed else '>'} 25% -> ", end="")
print("PASSED" if gate_passed else "NOT PASSED -- annotation budget reopened, evidence-based")
if not gate_passed:
    print("Decision (plan.md Step 18b DECISION, 2026-08-10): train set expanded 105 -> 122")
    print("pages, targeted at the chapters the repaired reader still failed hardest on --")
    print("ch04_elem_transcend +10, ch24_combinatorial +4, ch29_laplace +3 -- every added")
    print("page is one the repaired reader actually failed on, not an arbitrary pick.")


STEP 18B -- THE FOUR BEFORE/AFTER NUMBERS (plan.md's own gate)
metric                                    BEFORE (Step 16)  AFTER (this run)
------------------------------------------------------------------------------
no-output rate                                       42.9%             28.5%
median page coverage vs PDF text                     28.0%             43.7%
book-wide word coverage vs PDF text                  15.1%             28.3%
as_p0360 precision                                   0.813             0.458
as_p0360 recall                                      0.280             0.833
------------------------------------------------------------------------------

GATE (plan.md Step 18b): no-output rate > 25% -> NOT PASSED -- annotation budget reopened, evidence-based
Decision (plan.md Step 18b DECISION, 2026-08-10): train set expanded 105 -> 122
pages, targeted at the chapters the repaired reader still failed hardest on --
ch04_elem_transcend +10, ch24_combinatorial +4, ch2

In [3]:
# IMPLEMENT: run OCR quality + one retrieval, end to end
